# 三体问题数值求解（作业周练习）

**课程内容**：第六节 常微分方程的数值求解（欧拉法、梯形法、RK4）
**顺序**：先完成单体、二体、三体基本任务，再复现 figure-8；积分器长期对比列为附录。

从项目根目录顺序运行。独立参考数据在 `data/threebody_reference/`，生成命令与配置见 README。所有时间与状态量采用本题无量纲单位。

## 题目

1. **单体问题**：行星固定在原点，卫星受万有引力 $\ddot{\mathbf{r}} = -GM\mathbf{r}/r^3$。编程求解，给定至少一组定解条件，给出轨道图像，结合轨道运动原理分析正确性。（例如初始位置 $(0,2)$、初始速度 $(1,0)$，取 $g=1$）
2. **二体问题**：$m_1=0.3,\ m_2=0.03$，初始条件 $(x_1,y_1)=(2,2),\ (v_{x1},v_{y1})=(0.2,-0.2),\ (x_2,y_2)=(0,0),\ (v_{x2},v_{y2})=(-0.01,0.01)$。给出数值解、轨迹，验证正确性，观察微小初始条件改变的影响。
3. **三体问题**：$m_1=0.3,\ m_2=m_3=0.03$，$(x_1,y_1)=(2,2),\ (v_1)=(0.2,-0.2),\ (x_2,y_2)=(0,0),\ (v_2)=(0,0),\ (x_3,y_3)=(-2,-2),\ (v_3)=(-0.2,0.2)$。给出轨迹，微扰观察变化，**调研文献给出验证依据**。
4. **复现 Moore (1993) 的 figure-8 解**。

## 一、理论模型与验证原则

对质量为 $m_i$ 的质点，令 $\mathbf r_i$ 为位置：
$$\ddot{\mathbf r}_i=G\sum_{j\ne i}m_j\frac{\mathbf r_j-\mathbf r_i}{|\mathbf r_j-\mathbf r_i|^3}.$$
状态按 `[所有位置; 所有速度]` 排列。固定中心天体的单体问题另用 $\ddot{\mathbf r}=-GM\mathbf r/r^3$。

本笔记采用**平面模型**。检查能量 $E$、动量的两个分量 $P_x,P_y$、角动量标量 $L_z$，以及质心直线运动：
$$\mathbf R_{cm}(t)=\mathbf R_{cm}(0)+\frac{\mathbf P(0)}{\sum m_i}t.$$
守恒误差小是必要的数值诊断，**不是轨迹正确的充分条件**。本笔记还检查解析解、独立积分、步长收敛和时间反演；所有结论限定在已测试的时间区间。


## 二、数值方法

**课程方法（第六节）**：
- 显式欧拉法（一阶）
- 显式梯形法 / Heun（二阶）
- 四阶 Runge–Kutta（RK4）—— 作业主用方法

**拓展方法 ①：Leapfrog（velocity Verlet，辛积分器）**

$$v_{n+1/2} = v_n + \tfrac{h}{2}a(r_n), \qquad r_{n+1} = r_n + h\,v_{n+1/2}, \qquad v_{n+1} = v_{n+1/2} + \tfrac{h}{2}a(r_{n+1})$$

二阶精度，但**保辛**：在固定小步长、轨道光滑等适用条件下，常有较好的长期能量表现，且方法时间对称；仍可能存在相位误差。长时表现需实测。

**拓展方法 ②：自适应 RK45（Dormand–Prince）**

Embedded RK 对：同一步长下同时算 5 阶解与 4 阶嵌入解，差值估计局部误差，自动放大/缩小 $h$ 使误差落在容差内。非均匀轨道（近距高速）省步数、远距平滑时加大步长。

## 三、实现

### 3.1 ODE 求解器库

In [ ]:
# 三体问题数值求解 - 作业周练习
# 课程方法（第六节）+ 两个拓展方法（辛积分、自适应 RK45）

using LinearAlgebra, Plots, Printf, Statistics
gr()


# 显式欧拉
step_euler(f, y, t, h) = y + h * f(y, t)

# 显式梯形法（Heun）
function step_trapezoid(f, y, t, h)
    k1 = f(y, t)
    yp = y + h*k1
    y + h/2 * (k1 + f(yp, t+h))
end

# 四阶 Runge-Kutta
function step_rk4(f, y, t, h)
    k1 = f(y, t)
    k2 = f(y + h/2*k1, t + h/2)
    k3 = f(y + h/2*k2, t + h/2)
    k4 = f(y + h*k3, t + h)
    y + h/6 * (k1 + 2k2 + 2k3 + k4)
end

function integrate(f, y0, tspan; method=:rk4, h=0.01)
    t0, t1 = tspan
    n = max(2, round(Int, (t1-t0)/h))
    h = (t1-t0)/n
    t = collect(range(t0, t1; length=n+1))
    Y = zeros(eltype(y0), length(y0), n+1)
    Y[:,1] = y0
    step! = method == :euler ? step_euler :
            method == :trapezoid ? step_trapezoid :
            method == :rk4 ? step_rk4 :
            error("unknown method $method")
    for i in 1:n
        Y[:,i+1] = step!(f, Y[:,i], t[i], h)
    end
    return t, Y
end

# Dormand-Prince RK45（自适应步长）
# 此教学实现每步重算全部阶段，未采用 FSAL 缓存。
function dp_rk45(f, y0, tspan; h0=0.01, atol=1e-9, rtol=1e-8, hmin=1e-12, hmax=0.5)
    tspan[2] >= tspan[1] || throw(ArgumentError("仅支持正向时间区间"))
    0 < hmin <= h0 <= hmax || throw(ArgumentError("要求 0 < hmin <= h0 <= hmax"))
    atol > 0 && rtol >= 0 || throw(ArgumentError("要求 atol > 0 且 rtol >= 0"))
    # Dormand-Prince 系数
    a2=1/5; a3=3/10; a4=4/5; a5=8/9; a6=1.0
    b21=1/5
    b31=3/40; b32=9/40
    b41=44/45; b42=-56/15; b43=32/9
    b51=19372/6561; b52=-25360/2187; b53=64448/6561; b54=-212/729
    b61=9017/3168; b62=-355/33; b63=46732/5247; b64=49/176; b65=-5103/18656
    # 七阶段 Dormand–Prince 5(4) 嵌入式对
    b71=35/384; b72=0; b73=500/1113; b74=125/192; b75=-2187/6784; b76=11/84
    # 五阶解与四阶嵌入解之差用于估计误差
    # Error: difference between 5th and 4th order embedded
    # 4th order coefficients (e coefficients)
    e1=71/57600; e2=0; e3=-71/16695; e4=71/1920; e5=-17253/339200; e6=22/525; e7=-1/40

    function dp_step(y, t, h)
        k1 = f(y, t)
        k2 = f(y + h*(b21*k1), t + a2*h)
        k3 = f(y + h*(b31*k1 + b32*k2), t + a3*h)
        k4 = f(y + h*(b41*k1 + b42*k2 + b43*k3), t + a4*h)
        k5 = f(y + h*(b51*k1 + b52*k2 + b53*k3 + b54*k4), t + a5*h)
        k6 = f(y + h*(b61*k1 + b62*k2 + b63*k3 + b64*k4 + b65*k5), t + a6*h)
        # 五阶解与终点导数
        y_new = y + h*(b71*k1 + b73*k3 + b74*k4 + b75*k5 + b76*k6)
        k7 = f(y_new, t + h)
        # Error estimate (difference between 5th and 4th)
        err = h*(e1*k1 + e3*k3 + e4*k4 + e5*k5 + e6*k6 + e7*k7)
        return y_new, k7, err
    end

    t0, t1 = tspan
    t = Float64[t0]
    Y = [copy(y0)]
    h = h0
    while t[end] < t1
        y_old = Y[end]
        t_old = t[end]
        if t_old + h > t1
            h = t1 - t_old
        end
        t_old + h > t_old || error("步长无法推进浮点时间")
        y_new, _, err = dp_step(y_old, t_old, h)
        # 误差控制
        scale = atol .+ rtol .* max.(abs.(y_old), abs.(y_new))
        err_norm = maximum(abs.(err ./ scale))
        isfinite(err_norm) && all(isfinite,y_new) || error("积分出现非有限状态或误差")
        if err_norm <= 1.0
            # 接受步
            push!(t, t_old + h)
            push!(Y, y_new)
            # 调整步长
            if err_norm == 0
                h = min(hmax, h*5)
            else
                h = min(hmax, max(hmin, 0.9*h*err_norm^(-1/5)))
            end
        else
            # 拒绝步；无法进一步缩步时显式失败，避免无限循环。
            h <= hmin && error("最小步长下仍无法满足容差")
            h = max(hmin, 0.5*h*err_norm^(-1/5))
        end
        h = min(hmax, h)
    end
    # 不规则回保存：把最后点强制对齐到 t1
    if t[end] > t1
        # 重新做一步到 t1
        h = t1 - t[end-1]
        y_new, _, _ = dp_step(Y[end-1], t[end-1], h)
        t[end] = t1
        Y[end] = y_new
    end
    # 转换为矩阵
    T = copy(t)  # 自适应采样时间必须与 Mat 的每列一一对应
    Mat = zeros(length(y0), length(t))
    for (i, y) in enumerate(Y)
        Mat[:,i] = y
    end
    return T, Mat, Y, t
end

### 3.2 N 体动力学与守恒律

In [ ]:
# 状态 y = [x1,y1, x2,y2, ..., xN,yN, vx1,vy1, vx2,vy2, ..., vxN,vyN]
# 即 y[1:2N] 是位置，y[2N+1:4N] 是速度

function nbody_accel!(accel, pos, masses, G)
    N = length(masses)
    fill!(accel, 0.0)
    @inbounds for i in 1:N, j in 1:N
        i == j && continue
        dx = pos[2j-1] - pos[2i-1]
        dy = pos[2j]   - pos[2i]
        r2 = dx^2 + dy^2
        r = sqrt(r2)
        f = G * masses[j] / (r2*r)
        accel[2i-1] += f*dx
        accel[2i]   += f*dy
    end
    return accel
end

function nbody_f(y, t, masses, G)
    N = length(masses)
    pos = y[1:2N]; vel = y[2N+1:4N]
    a = zeros(2N)
    nbody_accel!(a, pos, masses, G)
    return [vel; a]
end

# Leapfrog step（用于 symplectic integrator）
function leapfrog_step(y, masses, G, h)
    N = length(masses)
    pos = y[1:2N]; vel = y[2N+1:4N]
    a = zeros(2N)
    nbody_accel!(a, pos, masses, G)
    vel_half = vel + h/2 * a
    pos_new = pos + h * vel_half
    nbody_accel!(a, pos_new, masses, G)
    vel_new = vel_half + h/2 * a
    return [pos_new; vel_new]
end

function integrate_leapfrog(y0, tspan, masses, G; h=0.01)
    t0, t1 = tspan
    n = max(2, round(Int, (t1-t0)/h))
    h = (t1-t0)/n
    t = collect(range(t0, t1; length=n+1))
    Y = zeros(eltype(y0), length(y0), n+1)
    Y[:,1] = y0
    for i in 1:n
        Y[:,i+1] = leapfrog_step(Y[:,i], masses, G, h)
    end
    return t, Y
end

function total_energy(y::AbstractVector, masses, G)
    N = length(masses)
    pos = y[1:2N]; vel = y[2N+1:4N]
    E_kin = 0.0
    for i in 1:N
        vx = vel[2i-1]; vy = vel[2i]
        E_kin += 0.5 * masses[i] * (vx^2 + vy^2)
    end
    E_pot = 0.0
    for i in 1:N, j in i+1:N
        dx = pos[2j-1] - pos[2i-1]
        dy = pos[2j]   - pos[2i]
        E_pot -= G * masses[i] * masses[j] / sqrt(dx^2+dy^2)
    end
    return E_kin + E_pot
end

total_momentum(y, masses) = begin
    N = length(masses)
    vel = y[2N+1:4N]
    Px = sum(masses[i]*vel[2i-1] for i in 1:N)
    Py = sum(masses[i]*vel[2i]   for i in 1:N)
    [Px, Py]
end

total_angular_momentum(y, masses) = begin
    N = length(masses)
    pos = y[1:2N]; vel = y[2N+1:4N]
    L = 0.0
    for i in 1:N
        L += masses[i] * (pos[2i-1]*vel[2i] - pos[2i]*vel[2i-1])
    end
    L
end

## 四、案例 1：单体问题（Kepler 轨道）

行星固定在原点，$G=1$，并**额外选择中心天体质量 $M=1$**，故 $GM=1$，卫星方程 $\ddot{\mathbf{r}} = -\mathbf{r}/r^3$。

**先做物理预判**：比能（单位卫星质量的能量）$E=\tfrac12 v^2 - \tfrac1r$ 决定轨道类型（活力公式 $v_{esc}=\sqrt{2/r}$）：

- 题目初值 $(0,2)$、$v=(1,0)$：$r=2 \Rightarrow v_{esc}=\sqrt{2/2}=1$，**速度恰好等于逃逸速度，$E=0$ → 抛物线轨道**（不闭合，飞向无穷远）
- $v<1 \Rightarrow E<0$：椭圆（闭合）
- $v>1 \Rightarrow E>0$：双曲线（飞离）

In [ ]:
println("="^60)
println("案例 1: 单体（Kepler 轨道，行星固定在原点）")
println("="^60)

# 单体：m1=1 受 m2 吸引（m2 固定在原点）
# 等效于：x'' = -GM x/(x²+y²)^(3/2)，取 GM=1
# 初始 (0, 2), v=(1, 0)
G = 1.0
m_single = [1.0]  # 仅卫星质量（行星质量纳入 GM）

# 加速函数：a = -G*M*(r)/|r|^3，这里 M = 行星（虚拟）
# 状态 y = [x, y, vx, vy]
function kepler_f(y, t, M, G)
    x, y_pos, vx, vy = y[1], y[2], y[3], y[4]
    r3 = (x^2 + y_pos^2)^1.5
    [vx, vy, -G*M*x/r3, -G*M*y_pos/r3]
end

# 用 m_eff=1（即 GM=1）
f_single = (y, t) -> kepler_f(y, t, 1.0, G)
y0_single = [0.0, 2.0, 1.0, 0.0]

# 解析：能量 E = ½v² - 1/r，角动量 L = x·vy - y·vx
E0_single = 0.5*(1^2 + 0) - 1.0/2.0
L0_single = 0.0*0.0 - 2.0*1.0
@printf("  初始能量 E₀ = %.6f, 角动量 L₀ = %.6f\n", E0_single, L0_single)

# 期望：E₀ = 0（抛物线/椭圆边界），半长轴 a = -1/(2E) = ∞ 时偏心率 e = 1
# 实际：E=0 → 抛物线轨道（双曲线/抛物线边界），不闭合

# 用 RK4 求解（短时间，验证守恒律）
tspan_s = (0.0, 50.0)
t_s, Y_s = integrate(f_single, y0_single, tspan_s; method=:rk4, h=0.001)
E_s = [0.5*(Y_s[3,i]^2+Y_s[4,i]^2) - 1.0/sqrt(Y_s[1,i]^2+Y_s[2,i]^2) for i in 1:length(t_s)]
L_s = [Y_s[1,i]*Y_s[4,i] - Y_s[2,i]*Y_s[3,i] for i in 1:length(t_s)]
@printf("  RK4 h=0.001, 步数 %d\n", length(t_s)-1)
@printf("  能量最大漂移: %.3e\n", maximum(abs.(E_s .- E0_single)))
@printf("  角动量最大漂移: %.3e\n", maximum(abs.(L_s .- L0_single)))

# 单体轨道图
mkpath("tmp/threebody")
p1 = plot(Y_s[1,:], Y_s[2,:]; lw=1, label="RK4 trajectory",
          xlabel="x", ylabel="y", title="Kepler orbit (E=0, parabolic-like)",
          aspect_ratio=:equal)
scatter!(p1, [0.0], [0.0]; ms=8, color=:red, label="Planet (origin)")
savefig(p1, "tmp/threebody/kepler_orbit.png")
println("  图像: tmp/threebody/kepler_orbit.png")
println()

# --- 单体补充：能量符号决定轨道类型（椭圆/抛物线/双曲线）---
# 逃逸速度 v_esc = sqrt(2GM/r)，在 r=2 处 v_esc = 1.0
# 题目给的 v=1.0 恰好等于逃逸速度 → E=0 抛物线；再对比 E<0 椭圆、E>0 双曲线
println("  轨道类型对比（同一初始位置 r=(0,2)，不同初速度）:")
v_ell, v_par, v_hyp = 0.5, 1.0, 1.3
for (vx, name) in [(v_ell, "椭圆 E<0"), (v_par, "抛物线 E=0"), (v_hyp, "双曲线 E>0")]
    y0 = [0.0, 2.0, vx, 0.0]
    E = 0.5*vx^2 - 1.0/2.0
    tspan_k = (0.0, vx <= 1.0 ? 60.0 : 20.0)
    t_k, Y_k = integrate(f_single, y0, tspan_k; method=:rk4, h=0.001)
    @printf("    %-12s v=%.2f  E=%.4f  a=%.3f\n", name, vx, E, -1.0/(2E))
end

# 椭圆轨道单独放大图（清晰展示闭合轨道）
y0_ell = [0.0, 2.0, v_ell, 0.0]
t_ell, Y_ell = integrate(f_single, y0_ell, (0.0, 60.0); method=:rk4, h=0.001)
p1c = plot(Y_ell[1,:], Y_ell[2,:]; lw=1.5, color=:royalblue, label="v=0.5, E=-0.375",
           xlabel="x", ylabel="y", title="Elliptical orbit (E<0, closed)",
           aspect_ratio=:equal)
scatter!(p1c, [0.0], [0.0]; ms=8, color=:black, label="Planet (origin)")
scatter!(p1c, [Y_ell[1,1]], [Y_ell[2,1]]; ms=5, color=:orange, label="start (0,2)")
savefig(p1c, "tmp/threebody/kepler_ellipse.png")
println("  图像: tmp/threebody/kepler_ellipse.png")

# 三轨对比图（限制显示范围，避免抛物线/双曲线把椭圆压扁）
p1b = plot(; xlabel="x", ylabel="y", title="Orbit type vs energy (single body, GM=1)",
           aspect_ratio=:equal, legend=:bottomright, xlims=(-3, 12), ylims=(-4, 4))
for (vx, lbl, col) in [(v_ell, "ellipse E<0", :royalblue),
                       (v_par, "parabola E=0", :red),
                       (v_hyp, "hyperbola E>0", :darkgreen)]
    y0 = [0.0, 2.0, vx, 0.0]
    tspan_k = (0.0, vx <= 1.0 ? 60.0 : 20.0)
    t_k, Y_k = integrate(f_single, y0, tspan_k; method=:rk4, h=0.001)
    plot!(p1b, Y_k[1,:], Y_k[2,:]; lw=1.5, label=lbl, color=col)
end
scatter!(p1b, [0.0], [0.0]; ms=8, color=:black, label="Planet (origin)")
savefig(p1b, "tmp/threebody/kepler_orbit_types.png")
println("  图像: tmp/threebody/kepler_orbit_types.png")
println()
# 直接显示本次生成的图，避免正文嵌入过期图片。
display(p1)
display(p1b)
display(p1c)


**单体结果的解释**

在 $G=M=1$ 下，题目初值的比能 $\varepsilon=v^2/2-1/r=0$，角动量 $\ell=-2$，故轨道为抛物线，而非闭合椭圆。
其解析参数式为
$$x=4D,\quad y=2-2D^2,\quad t=4(D+D^3/3),$$
因此 $x^2=16-8y$。后面的基本任务验证单元会同时比较位置与速度，避免只看轨道形状而忽略到达时间。
补充的切向初速度 $v=0.5$ 对应椭圆，$v=1.3$ 对应双曲线；该判定基于当前固定位置和非零切向速度。


## 五、案例 2：二体问题

采用题目原始质量和全部初值。验证能量、两个动量分量、角动量和质心运动，并用独立 DOP853 解与步长减半对照。
二体相对运动满足 Kepler 方程，原坐标系另有质心匀速漂移。非混沌不等于两条轨迹的坐标差永远有界：微扰可能同时改变质心速度和轨道相位。


In [ ]:
println("="^60)
println("案例 2: 二体（m1=0.3, m2=0.03, G=1）")
println("="^60)

m2_body = [0.3, 0.03]
G = 1.0
# 状态：y = [x1,y1, x2,y2, vx1,vy1, vx2,vy2]
y0_2body = [2.0, 2.0,   # m1
            0.0, 0.0,   # m2
            0.2, -0.2,  # m1 速度
            -0.01, 0.01] # m2 速度
f_2body = (y, t) -> nbody_f(y, t, m2_body, G)

t0_2 = 0.0
t1_2 = 20.0
t_2, Y_2 = integrate(f_2body, y0_2body, (t0_2, t1_2); method=:rk4, h=0.001)

E_2 = [total_energy(Y_2[:,i], m2_body, G) for i in 1:length(t_2)]
P_2 = [total_momentum(Y_2[:,i], m2_body) for i in 1:length(t_2)]
L_2 = [total_angular_momentum(Y_2[:,i], m2_body) for i in 1:length(t_2)]

@printf("  RK4 h=0.001, 步数 %d\n", length(t_2)-1)
@printf("  初始能量 E₀ = %.6f, 末态 = %.6f, 漂移 = %.3e\n",
        E_2[1], E_2[end], abs(E_2[end]-E_2[1]))
@printf("  初始动量 P₀ = (%.6f, %.6f), 末态 = (%.6f, %.6f)\n",
        P_2[1]..., P_2[end]...)
@printf("  初始角动量 L₀ = %.6f, 末态 = %.6f, 漂移 = %.3e\n",
        L_2[1], L_2[end], abs(L_2[end]-L_2[1]))

p2 = plot(Y_2[1,:], Y_2[2,:]; lw=1, label="m₁=0.3",
          xlabel="x", ylabel="y", title="Two-body trajectories (20s)",
          aspect_ratio=:equal)
plot!(p2, Y_2[3,:], Y_2[4,:]; lw=1, label="m₂=0.03")
savefig(p2, "tmp/threebody/two_body.png")

# 能量守恒曲线
p2b = plot(t_2, (E_2 .- E_2[1])./abs(E_2[1]); lw=1, label="relative ΔE",
           xlabel="t", ylabel="ΔE / |E₀|", title="Energy conservation (two-body)")
savefig(p2b, "tmp/threebody/two_body_energy.png")
println("  图像: tmp/threebody/two_body.png, two_body_energy.png")
println()

# 微小扰动：vy₂ 增加 1e-6，即该分量的 0.01%。
delta_v2 = 1e-6
y0_2body_pert = copy(y0_2body)
y0_2body_pert[8] += delta_v2
t_2p, Y_2p = integrate(f_2body, y0_2body_pert, (t0_2, t1_2); method=:rk4, h=0.001)
p2c = plot(Y_2[1,:], Y_2[2,:]; lw=1, label="original m₁",
           xlabel="x", ylabel="y", title="Two-body perturbation (delta vy2=1e-6)",
           aspect_ratio=:equal)
plot!(p2c, Y_2[3,:], Y_2[4,:]; lw=1, label="original m₂")
plot!(p2c, Y_2p[1,:], Y_2p[2,:]; lw=1, ls=:dash, label="perturbed m₁")
plot!(p2c, Y_2p[3,:], Y_2p[4,:]; lw=1, ls=:dash, label="perturbed m₂")
savefig(p2c, "tmp/threebody/two_body_perturb.png")
println("  扰动图像: tmp/threebody/two_body_perturb.png")
println()
display(p2)
display(p2b)
display(p2c)


**二体结果的解释**

总动量为 $(0.0597,-0.0597)$，质心并不静止。相对坐标的比能为负，理论轨道为椭圆；半长轴约 2.273585、偏心率约 0.244039、周期约 37.4964。本例 $0\le t\le20$ 尚不足一个完整周期。
微扰采用 $\Delta v_{y2}=10^{-6}$。随后报告去除各自质心运动后的位置差，并比较 $10^{-8}$ 扰动及步长减半结果。小扰动曲线在普通轨道图上可能重叠，这是预期现象。


## 六、案例 3：三体问题

保持老师给定的质量和初值。由于总动量 $(0.054,-0.054)$ 非零，质心应沿
$$\mathbf R_{cm}(t)=(1.5,1.5)+(0.15,-0.15)t$$
运动。

本例验证包括：全步长守恒误差、质心直线误差、独立积分结果、步长减半和时间反演。
时间反演采用**末态速度取反后正向积分相同时长，再把速度取反**，与初始完整状态比较。

[Hairer、Lubich 与 Wanner](https://www.unige.ch/~hairer/preprints/gniverlet.html) 讨论几何积分的可逆性与长期误差；[Rein 与 Spiegel](https://arxiv.org/abs/1409.4779) 展示引力积分器的精度评估。本笔记实际独立对照采用 [SciPy DOP853](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html)，未将文献结论或 IAS15 的性能视作本例的实测结果。


In [ ]:
println("="^60)
println("案例 3: 三体（m1=0.3, m2=m3=0.03, G=1）")
println("="^60)

m3_body = [0.3, 0.03, 0.03]
G = 1.0
y0_3body = [2.0, 2.0,    # m1
            0.0, 0.0,    # m2
           -2.0,-2.0,    # m3
            0.2,-0.2,    # m1 速度
            0.0, 0.0,    # m2 速度
           -0.2, 0.2]    # m3 速度

# 总动量检查：m1*v1 + m2*v2 + m3*v3 = 0.3*(0.2,-0.2) + 0.03*(0,0) + 0.03*(-0.2,0.2)
P_init = 0.3*[0.2,-0.2] + 0.03*[0.0,0.0] + 0.03*[-0.2,0.2]
@printf("  初始总动量 P₀ = (%.4f, %.4f)\n", P_init...)

f_3body = (y, t) -> nbody_f(y, t, m3_body, G)
t0_3, t1_3 = 0.0, 30.0
t_3, Y_3 = integrate(f_3body, y0_3body, (t0_3, t1_3); method=:rk4, h=0.001)

E_3 = [total_energy(Y_3[:,i], m3_body, G) for i in 1:length(t_3)]
P_3 = [total_momentum(Y_3[:,i], m3_body) for i in 1:length(t_3)]
L_3 = [total_angular_momentum(Y_3[:,i], m3_body) for i in 1:length(t_3)]

@printf("  RK4 h=0.001, 步数 %d\n", length(t_3)-1)
@printf("  初始能量 E₀ = %.6f\n", E_3[1])
@printf("  末态能量 %.6f, 漂移 %.3e\n", E_3[end], abs(E_3[end]-E_3[1]))
@printf("  初始总动量 (%.6f, %.6f), 末态 (%.6f, %.6f)\n", P_3[1]..., P_3[end]...)
@printf("  初始角动量 L₀ = %.6f, 末态 %.6f, 漂移 %.3e\n", L_3[1], L_3[end], abs(L_3[end]-L_3[1]))

p3 = plot(Y_3[1,:], Y_3[2,:]; lw=1.2, label="m₁=0.3",
          xlabel="x", ylabel="y", title="Three-body trajectories (30s, RK4)",
          aspect_ratio=:equal)
plot!(p3, Y_3[3,:], Y_3[4,:]; lw=1, label="m₂=0.03")
plot!(p3, Y_3[5,:], Y_3[6,:]; lw=1, label="m₃=0.03")
savefig(p3, "tmp/threebody/three_body.png")

p3b = plot(t_3, (E_3 .- E_3[1])./abs(E_3[1]); lw=1, label="relative ΔE",
           xlabel="t", ylabel="ΔE / |E₀|", title="Energy conservation (three-body)")
savefig(p3b, "tmp/threebody/three_body_energy.png")
println("  图像: tmp/threebody/three_body.png, three_body_energy.png")

# 有限时间初值敏感性：扰动 m3 的 x' 1e-6
y0_3body_pert = copy(y0_3body)
y0_3body_pert[11] = -0.2 + 1e-6   # m3 的 vx 微扰 1e-6
t_3p, Y_3p = integrate(f_3body, y0_3body_pert, (t0_3, t1_3); method=:rk4, h=0.001)
# 两解距离
diff_m1 = sqrt.((Y_3[1,:] .- Y_3p[1,:]).^2 .+ (Y_3[2,:] .- Y_3p[2,:]).^2)
p3c = plot(t_3, log10.(max.(diff_m1,eps(Float64))); lw=1,
           xlabel="t", ylabel="log₁₀(Δposition m₁)",
           title="Initial-condition sensitivity: delta vx3=1e-6")
savefig(p3c, "tmp/threebody/three_body_chaos.png")
@printf("  末态 |Δm₁| = %.3e (从 1e-6 扰动)\n", diff_m1[end])
println("  微扰图像: tmp/threebody/three_body_chaos.png")
println()
display(p3)
display(p3b)
display(p3c)


**三体结果的解释与边界**

在当前 $0\le t\le30$ 内，第三体与另外两体的距离明显增加；不能把图像描述为“整体束缚”，也不能仅凭有限时间结果断定最终逃逸结局。
$\Delta v_{x3}=10^{-6}$ 后，第一个天体末态位置差约为 $4.7\times10^{-7}$。位置差与速度扰动单位不同，不能直接相除作为无量纲放大率。
下面同时报告全系统去质心位置差、两种扰动幅度和数值误差。**这些实验用于描述有限时间敏感性，不构成混沌或指数增长的证明。**


## 七、基本任务：可重复的验证实验

先完成并检查本节，再进入 figure-8。参考 CSV 由独立 Python 成对引力实现与 DOP853 生成。采样间隔为 0.1；守恒与最近距离在 RK4 每个时间步检查。三组步长细化、时间反演及扰动数据都由本单元实算。

In [ ]:
# Diagnostics shared by the notebook and regression tests.
using LinearAlgebra, DelimitedFiles, Printf

function parabolic_state(t)
    D=2sinh(asinh(3t/8)/3)
    [4D, 2-2D^2, 1/(1+D^2), -D/(1+D^2)]
end

function centers(Y, masses)
    N=length(masses)
    sum(masses[i].*Y[2i-1:2i,:] for i in 1:N)/sum(masses)
end

function relative_positions(Y, masses)
    N=length(masses); C=centers(Y,masses)
    reduce(vcat,[Y[2i-1:2i,:]-C for i in 1:N])
end

function orbit_diagnostics(t,Y,masses;G=1.0)
    N=length(masses); nt=length(t)
    P=sum(masses[i].*Y[2N+2i-1:2N+2i,:] for i in 1:N)
    C=centers(Y,masses)
    L=zeros(nt);E=zeros(nt);rmin=fill(Inf,nt)
    for k in 1:nt
        for i in 1:N
            x,y=Y[2i-1:2i,k];vx,vy=Y[2N+2i-1:2N+2i,k]
            E[k]+=masses[i]*(vx^2+vy^2)/2
            L[k]+=masses[i]*(x*vy-y*vx)
            for j in i+1:N
                r=norm(Y[2i-1:2i,k]-Y[2j-1:2j,k])
                E[k]-=G*masses[i]*masses[j]/r
                rmin[k]=min(rmin[k],r)
            end
        end
    end
    predicted=C[:,1].+(P[:,1]/sum(masses))*(t.-t[1])'
    (;energy=E,angular=L,momentum=P,center=C,min_distance=rmin,
      energy_error=maximum(abs.(E.-E[1])),angular_error=maximum(abs.(L.-L[1])),
      momentum_error=maximum(abs.(P.-P[:,1])),cm_error=maximum(abs.(C.-predicted)),
      internal_energy=E[1]-sum(abs2,P[:,1])/(2sum(masses)))
end

function reversal_error(f,y0,T;h=.001)
    N=length(y0)÷4
    _,Y=integrate(f,y0,(0.,T);h=h)
    z=copy(Y[:,end]);z[2N+1:end].*=-1
    _,Z=integrate(f,z,(0.,T);h=h)
    zback=copy(Z[:,end]);zback[2N+1:end].*=-1
    maximum(abs.(zback-y0))
end

function read_reference(name)
    path=joinpath("data","threebody_reference",name*".csv")
    isfile(path) || error("缺少独立参考数据 $(path)；请在项目根目录运行笔记，或先运行 scripts/threebody_reference.py")
    a=readdlm(path,',',Float64)
    a[:,1],permutedims(a[:,2:end])
end

function aligned_solution(f,y0,tref;h)
    T=tref[end]-tref[1]
    t,Y=integrate(f,y0,(tref[1],tref[end]);h=h)
    indices=round.(Int,(tref.-tref[1])./(t[2]-t[1])).+1
    maximum(abs.(t[indices]-tref)) < 1e-9 || error("比较网格未对齐")
    t,Y,indices
end

function basic_validation()
    cases=[("single",[1.],[0.,2.,1.,0.],50.),
           ("two",[.3,.03],[2.,2.,0.,0.,.2,-.2,-.01,.01],20.),
           ("three",[.3,.03,.03],[2.,2.,0.,0.,-2.,-2.,.2,-.2,0.,0.,-.2,.2],30.)]
    results=Dict{String,Any}()
    for (name,m,y0,T) in cases
        f=name=="single" ? ((z,t)->kepler_f(z,t,1.,1.)) : ((z,t)->nbody_f(z,t,m,1.))
        tr,ref=read_reference(name)
        hs=[.1,.05,.025,.0125,.001,.0005]
        errors=Float64[]; sampled=Matrix{Float64}[]
        for h in hs
            _,Y,idx=aligned_solution(f,y0,tr;h=h)
            push!(sampled,Y[:,idx]);push!(errors,maximum(abs.(Y[:,idx]-ref)))
        end
        t,Y,idx=aligned_solution(f,y0,tr;h=.001)
        reverse=reversal_error(f,y0,T;h=.001)
        println("\n",name,": 同时刻所有状态分量最大绝对差（无量纲）")
        for (h,e) in zip(hs,errors);@printf("  h=%-8g 与独立 DOP853 差 = %.3e\n",h,e);end
        @printf("  h=0.001 减半差 = %.3e；时间反演误差 = %.3e\n",maximum(abs.(sampled[end-1]-sampled[end])),reverse)
        if name=="single"
            exact=reduce(hcat,parabolic_state.(tr))
            analytic=maximum(abs.(sampled[end-1]-exact))
            @printf("  对抛物线解析状态的最大误差 = %.3e\n",analytic)
            @assert analytic < 1e-8
            results[name]=(;hs,errors,tr,ref,sampled,analytic)
        else
            d=orbit_diagnostics(t,Y,m)
            @printf("  全步检查 max|ΔE|=%.3e, max|ΔP|=%.3e, max|ΔL|=%.3e, 质心误差=%.3e\n",d.energy_error,d.momentum_error,d.angular_error,d.cm_error)
            @printf("  初始内部能量=%.9e；全步最小两体距离=%.6f\n",d.internal_energy,minimum(d.min_distance))
            perts=NamedTuple[]
            for delta in [1e-6,1e-8]
                yp=copy(y0);yp[name=="two" ? 8 : 11]+=delta
                _,Yp,ip=aligned_solution(f,yp,tr;h=.001)
                _,Ypf,ipf=aligned_solution(f,yp,tr;h=.0005)
                rel=relative_positions(Yp[:,ip],m)-relative_positions(sampled[end-1],m)
                dr=[norm(rel[:,k]) for k in eachindex(tr)]
                signal_error=maximum(abs.((Yp[:,ip]-sampled[end-1])-(Ypf[:,ipf]-sampled[end])))
                independent_error=NaN
                if delta==1e-6
                    tp,rp=read_reference(name*"_pert")
                    @assert maximum(abs.(tp-tr))<1e-12
                    independent_error=maximum(abs.((Yp[:,ip]-sampled[end-1])-(rp-ref)))
                    @printf("  扰动信号与独立 DOP853 的最大差 = %.3e\n",independent_error)
                    @assert independent_error < 1e-8
                end
                @printf("  Δv=%.1e：末态去质心位置差范数=%.6e，扰动信号减半差=%.3e\n",delta,dr[end],signal_error)
                push!(perts,(;delta,dr,signal_error,independent_error))
                @assert signal_error < 1e-9
            end
            @assert d.cm_error < 1e-9
            results[name]=(;hs,errors,tr,ref,sampled,d,t,Y,m,perts)
        end
        @assert errors[end-1] < 1e-8
        @assert reverse < 1e-8
    end
    results
end

function figure8_validation(y0,T;periods=20)
    periods==20 || throw(ArgumentError("独立参考数据覆盖固定 20 周期"))
    tr,R=read_reference("figure8")
    @assert abs(tr[end]-periods*T)<1e-10
    f=(z,t)->nbody_f(z,t,ones(3),1.)
    perperiod=600
    ns=[300,600,1200,2400,4800,9600]
    closure=Float64[];reference_errors=Float64[]
    for n in ns
        t,Y=integrate(f,y0,(0.,T);h=T/n)
        stride=max(1,n÷perperiod);ri=n<perperiod ? (1:2:perperiod+1) : (1:perperiod+1)
        push!(closure,maximum(abs.(Y[:,end]-y0)))
        push!(reference_errors,maximum(abs.(Y[:,1:stride:end]-R[:,ri])))
        @printf("steps/period=%5d, state closure=%.3e, independent difference=%.3e\n",n,closure[end],reference_errors[end])
    end
    n=2400
    t,Y=integrate(f,y0,(0.,periods*T);h=T/n)
    tf,Yf=integrate(f,y0,(0.,periods*T);h=T/(2n))
    idx=1:(n÷perperiod):length(t);idxf=1:(2n÷perperiod):length(tf)
    @assert maximum(abs.(t[idx]-tr))<1e-10
    state_delta=Y[:,idx]-R
    errors=[maximum(abs.(state_delta[:,k])) for k in eachindex(tr)]
    @assert maximum(abs.(Y[:,idx]-Yf[:,idxf])) < 1e-5
    # Convention for this notebook's body ordering: after T/3, (1,2,3) matches (3,1,2).
    perm=[5,6,1,2,3,4,11,12,7,8,9,10]
    shift=n÷3
    choreo=maximum(abs.(Y[:,shift+1:n+1]-Y[perm,1:n+1-shift]))
    d=orbit_diagnostics(t,Y,ones(3))
    period_indices=1:n:length(t)
    position_closure=[maximum(abs.(Y[1:6,k]-y0[1:6])) for k in period_indices]
    velocity_closure=[maximum(abs.(Y[7:12,k]-y0[7:12])) for k in period_indices]
    # Tangent projection of position error onto reference velocity: a local phase-time estimate.
    phase=[dot(state_delta[1:6,k],R[7:12,k])/sum(abs2,R[7:12,k]) for k in eachindex(tr)]
    transverse=[norm(state_delta[1:6,k]-phase[k]*R[7:12,k]) for k in eachindex(tr)]
    @printf("T/3 完整状态置换残差 = %.3e\n",choreo)
    @printf("20 周期 max|ΔE|=%.3e, max|ΔP|=%.3e, max|ΔL|=%.3e, min distance=%.6f\n",d.energy_error,d.momentum_error,d.angular_error,minimum(d.min_distance))
    @printf("20 周期末位置闭合=%.3e, 速度闭合=%.3e；与同初值独立轨迹最大差=%.3e\n",position_closure[end],velocity_closure[end],maximum(errors))
    @printf("同初值独立轨迹自身一周期闭合残差=%.3e（含给定初值和周期舍入影响）\n",maximum(abs.(R[:,601]-y0)))
    @assert choreo < 1e-5
    @assert closure[end] < 1e-6
    @assert maximum(errors) < 1e-5
    @assert minimum(d.min_distance) > .1
    (;ns,closure,reference_errors,tr,R,t,Y,errors,phase,transverse,d,position_closure,velocity_closure,choreo,T)
end


In [ ]:
using Markdown, TOML
println("独立参考容差从 rtol=1e-12 收紧至 3e-14 时的最大状态差（不是严格误差上界）：")
println(read(joinpath("data","threebody_reference","tolerance_checks.csv"),String))
basic_results = basic_validation()
pconv=plot(xscale=:log10,yscale=:log10,xlabel="step h",ylabel="max state difference",title="RK4 vs independent DOP853")
for name in ("single","two","three")
    d=basic_results[name]; plot!(pconv,d.hs,max.(d.errors,eps(Float64));marker=:circle,label=name)
end
display(pconv)
savefig(pconv,"tmp/threebody/basic_convergence.png")
for name in ("two","three")
    d=basic_results[name];rel=relative_positions(d.Y,d.m)
    p=plot(aspect_ratio=:equal,xlabel="x - Xcm",ylabel="y - Ycm",title=name*": center-of-mass frame")
    for i in eachindex(d.m);plot!(p,rel[2i-1,:],rel[2i,:];label="body $i");end
    display(p);savefig(p,"tmp/threebody/"*name*"_com.png")
    p=plot(yscale=:log10,xlabel="time",ylabel="relative position error / delta v",title=name*": finite-time response (time units)",left_margin=8Plots.mm,size=(800,450))
    for z in d.perts
        plot!(p,d.tr[2:end],max.(z.dr[2:end]./z.delta,eps(Float64));label="delta v=$(z.delta)")
    end
    display(p);savefig(p,"tmp/threebody/"*name*"_sensitivity.png")
end
d=basic_results["three"]
pdist=plot(xlabel="time",ylabel="separation",title="Three-body pair distances")
for (i,j) in [(1,2),(1,3),(2,3)]
    distances=[norm(d.Y[2i-1:2i,k]-d.Y[2j-1:2j,k]) for k in eachindex(d.t)]
    plot!(pdist,d.t,distances;label="r$(i)$(j)")
    @printf("r%d%d(T)=%.6f\n",i,j,distances[end])
end
display(pdist);savefig(pdist,"tmp/threebody/three_distances.png")

# Save compact numerical results and show a readable table.
basic_summary=Dict{String,Any}()
rows=["|案例|与独立解最大差|步长减半最大差|", "|---|---:|---:|"]
for name in ("single","two","three")
    d=basic_results[name]
    basic_summary[name]=Dict("reference_error"=>d.errors[end-1],"halving_difference"=>maximum(abs.(d.sampled[end-1]-d.sampled[end])))
    push!(rows,@sprintf("|%s|%.3e|%.3e|",name,basic_summary[name]["reference_error"],basic_summary[name]["halving_difference"]))
end
open(joinpath("data","threebody_basic_results.toml"),"w") do io;TOML.print(io,basic_summary);end
display(Markdown.parse(join(rows,"\n")))


**读表方式**：粗步长下预期 RK4 误差约按四次方缩小；很细步长下舍入误差和参考误差可能占主导。微扰曲线除以速度扰动后仍具有时间量纲，不是 Lyapunov 指数。两种扰动的响应接近只说明当前扰动范围内近似线性。所有自动断言仅验收此题给定案例与时间区间。

## 八、附加任务：复现 Moore 的 figure-8 解

Moore（1993）从拓扑与作用量角度研究周期轨道；Chenciner 与 Montgomery（2000）证明了等质量平面三体八字形周期轨道的存在。
这里复现该轨道，而不是重做存在性证明。采用常用的有限位数初值，$G=m_1=m_2=m_3=1$，周期近似 $T=6.32591398$。它是独立测试案例，**不沿用基本题的质量比**。
本笔记天体标签为“左侧、中心、右侧”，与一些文献排列不同；所有位置和速度按同一置换重排。

来源：[Moore 1993](https://doi.org/10.1103/PhysRevLett.70.3675)、[Chenciner–Montgomery 2000](https://arxiv.org/abs/math/0011268)、[常用初值与周期示例](https://juliagni.github.io/GeometricProblems.jl/latest/three_body_problem/)。


In [ ]:
println("="^60)
println("案例 4: figure-8 解（Moore 1993, 三等质量 m=1, G=1）")
println("="^60)

# 文献中常用的有限位数近似初值，不是精确小数解。
m8 = [1.0, 1.0, 1.0]
# Chenciner-Montgomery (2000) / Moore (1993) 标准初始条件（质心系，零角动量）
y0_8 = [
    -0.97000436,   0.24308753,   # m1 位置
     0.0,          0.0,          # m2 位置（原点）
     0.97000436,  -0.24308753,   # m3 位置
     0.4662036850, 0.4323657300, # m1 速度
    -0.9324073700,-0.8647314600, # m2 速度
     0.4662036850, 0.4323657300  # m3 速度
]

T8 = 6.32591398  # 一个周期
f_8 = (y, t) -> nbody_f(y, t, m8, G)

# 用 RK4 跑 1 个周期
h_8 = 0.001
n_8 = round(Int, T8/h_8)
t_8, Y_8 = integrate(f_8, y0_8, (0.0, T8); method=:rk4, h=h_8)
@printf("  RK4 h=%.3f 跑 1 周期 T=%.4f, 步数 %d\n", h_8, T8, n_8)

# 验证回到初始位置
err_close = maximum(abs.(Y_8[:,end] .- y0_8))
@printf("  周期末态 - 初始完整状态最大差: %.3e\n", err_close)

# 能量守恒
E_8 = [total_energy(Y_8[:,i], m8, G) for i in 1:length(t_8)]
@printf("  能量漂移: %.3e\n", maximum(abs.(E_8 .- E_8[1])))

# figure-8 轨迹图
p8 = plot(Y_8[1,:], Y_8[2,:]; lw=1.2, label="body 1",
          xlabel="x", ylabel="y", title="Figure-8 solution (Moore 1993)",
          aspect_ratio=:equal, size=(700, 700))
plot!(p8, Y_8[3,:], Y_8[4,:]; lw=1.2, label="body 2")
plot!(p8, Y_8[5,:], Y_8[6,:]; lw=1.2, label="body 3")
savefig(p8, "tmp/threebody/figure8.png")
println("  图像: tmp/threebody/figure8.png")
println()
display(p8)


**复现验收标准**

不能仅凭轨迹呈八字形认定复现完成。下列实验将检查：
1. 一周期后位置和速度同时接近初值；
2. $T/3$ 后完整状态按天体标签循环置换；
3. 多个步长与相同初值的独立解对照；
4. 20 周期的闭合偏差、能量误差及局部相位诊断；
5. 最小两体距离保持正值，当前计算未遇到碰撞奇点。

闭合误差同时含初值与周期舍入、积分离散和浮点误差；相同初值的独立积分有助于区分来源。此处不声称数学证明轨道稳定性。


In [ ]:
fig8_results=figure8_validation(y0_8,T8)
d=fig8_results
p8check=plot(layout=(2,2),size=(1250,850),left_margin=8Plots.mm,bottom_margin=6Plots.mm)
plot!(p8check[1],d.ns,d.closure;xscale=:log10,yscale=:log10,marker=:circle,label="closure to initial state",xlabel="steps / period",ylabel="max state error")
plot!(p8check[1],d.ns,d.reference_errors;marker=:square,label="vs same-IC DOP853")
plot!(p8check[2],0:20,max.(d.position_closure,eps(Float64));yscale=:log10,marker=:circle,label="position",xlabel="period count",ylabel="return error")
plot!(p8check[2],0:20,max.(d.velocity_closure,eps(Float64));marker=:square,label="velocity")
plot!(p8check[3],d.t./T8,max.(abs.(d.d.energy.-d.d.energy[1]),eps(Float64));yscale=:log10,label="absolute energy drift",xlabel="period count")
plot!(p8check[4],d.tr./T8,d.phase;label="local phase-time shift",xlabel="period count",ylabel="phase-time difference")
display(p8check);savefig(p8check,"tmp/threebody/figure8_validation.png")
# Show three labelled bodies at a common instant, avoiding coincident full-path ambiguity.
p8snap=plot(aspect_ratio=:equal,xlabel="x",ylabel="y",title="Figure-8: labels at T/12",legend=:outerright,size=(850,480),left_margin=6Plots.mm)
for i in 1:3
    plot!(p8snap,d.Y[2i-1,1:2401],d.Y[2i,1:2401];lw=1,label="path $i")
    scatter!(p8snap,[d.Y[2i-1,201]],[d.Y[2i,201]];ms=7,label="body $i at T/12")
end
display(p8snap);savefig(p8snap,"tmp/threebody/figure8_snapshot.png")

fig8_summary=Dict("period"=>T8,"one_period_closure"=>d.closure[end],"one_period_reference_difference"=>d.reference_errors[end],"choreography_residual"=>d.choreo,"twenty_period_reference_difference"=>maximum(d.errors),"twenty_period_position_closure"=>d.position_closure[end],"twenty_period_velocity_closure"=>d.velocity_closure[end],"energy_drift"=>d.d.energy_error,"minimum_distance"=>minimum(d.d.min_distance))
open(joinpath("data","figure8_results.toml"),"w") do io;TOML.print(io,fig8_summary);end


**figure-8 结果解释**：闭合误差与“相同初值独立轨迹的差异”应分开阅读。若步长继续缩小时后者降低，而前者趋于平台，说明仅减步长不能消除给定有限位数初值和周期带来的偏差。局部相位时间偏移由位置差在参考速度方向上的投影估计，适用于两轨迹接近的情况，不是精确全局相位。20 周期实验不等于无限时间稳定性结论。

## 九、可选附录：长时间积分器对比

同一步长对比仅展示方法行为，不能直接当作相同计算成本下的效率比较。

In [ ]:
println("="^60)
println("案例 5: 拓展方法对比（leapfrog 辛积分 vs RK4）")
println("="^60)

T_compare = 1000.0
h_c = 0.01

# RK4
t_c_rk4, Y_c_rk4 = integrate(f_8, y0_8, (0.0, T_compare); method=:rk4, h=h_c)
E_c_rk4 = [total_energy(Y_c_rk4[:,i], m8, G) for i in 1:length(t_c_rk4)]
err_rk4 = abs.(E_c_rk4 .- E_c_rk4[1]) / abs(E_c_rk4[1])

# Leapfrog
t_c_lf, Y_c_lf = integrate_leapfrog(y0_8, (0.0, T_compare), m8, G; h=h_c)
E_c_lf = [total_energy(Y_c_lf[:,i], m8, G) for i in 1:length(t_c_lf)]
err_lf = abs.(E_c_lf .- E_c_lf[1]) / abs(E_c_lf[1])

@printf("  长时间演化 T=%.0f, h=%.3f\n", T_compare, h_c)
@printf("  RK4 末态能量相对漂移: %.3e (最大值: %.3e)\n", err_rk4[end], maximum(err_rk4))
@printf("  Leapfrog 末态能量相对漂移: %.3e (最大值: %.3e)\n", err_lf[end], maximum(err_lf))

# 能量漂移对比
p_cmp = plot(t_c_rk4, max.(err_rk4,eps(Float64)); lw=1, yscale=:log10, label="RK4",
             xlabel="t", ylabel="|ΔE|/|E₀|", title="Energy drift: RK4 vs Leapfrog (T=$(T_compare))")
plot!(p_cmp, t_c_lf, max.(err_lf,eps(Float64)); lw=1, label="Leapfrog (symplectic)")
savefig(p_cmp, "tmp/threebody/compare_energy.png")

# 长时间 figure-8 轨迹（多周期）
p8_long = plot(Y_c_lf[1,:], Y_c_lf[2,:]; lw=0.5, label="body 1 (leapfrog, T=$(T_compare))",
               xlabel="x", ylabel="y", title="Figure-8 long-term (T=$(T_compare), leapfrog)",
               aspect_ratio=:equal, size=(700, 700))
plot!(p8_long, Y_c_lf[3,:], Y_c_lf[4,:]; lw=0.5, label="body 2")
plot!(p8_long, Y_c_lf[5,:], Y_c_lf[6,:]; lw=0.5, label="body 3")
savefig(p8_long, "tmp/threebody/figure8_long.png")
println("  图像: tmp/threebody/compare_energy.png, figure8_long.png")

# 自适应 RK45：演示自动步长
println()
println("="^60)
println("案例 6: 自适应 RK45（Dormand-Prince）")
println("="^60)
t_rk45, Y_rk45, Y_list, t_list = dp_rk45(f_8, y0_8, (0.0, 50.0);
                                          h0=0.01, atol=1e-9, rtol=1e-7)
n_rk45 = length(t_rk45)
@printf("  自适应 RK45: 实际步数 = %d (固定 h=0.01 需 %d 步)\n",
        n_rk45-1, round(Int, 50.0/0.01))
E_rk45 = [total_energy(Y_rk45[:,i], m8, G) for i in 1:n_rk45]
@printf("  能量漂移: %.3e\n", maximum(abs.(E_rk45 .- E_rk45[1])))
println()

println("="^60)
println("全部图像保存到: tmp/threebody/")
println("="^60)
display(p_cmp)
display(p8_long)


**附录解释**

固定步长下观察能量误差随时间的变化，同时须注意轨道相位误差。辛积分器不保证原始能量精确守恒，也不保证没有相位漂移。RK45 的接受步数不等于力计算次数或运行耗时，本例不据步数比例宣称加速倍数。

## 十、结论与参考文献

基本题和 figure-8 的数值证据由上面的验证单元直接生成。误差表比较的是同一时刻的无量纲状态分量；独立数值解也有误差，并非严格真值。
本例微扰实验不证明混沌。figure-8 的闭合、对称性和长期诊断支持有限精度下的数值复现，不等于新轨道发现或稳定性的数学证明。

1. Moore, C. (1993). *Braids in classical dynamics*. Physical Review Letters 70, 3675–3679. [DOI](https://doi.org/10.1103/PhysRevLett.70.3675)
2. Chenciner, A. & Montgomery, R. (2000). *A remarkable periodic solution of the three-body problem in the case of equal masses*. Annals of Mathematics 152, 881–901. [原文](https://arxiv.org/abs/math/0011268)
3. Hairer, E., Lubich, C. & Wanner, G. (2003). *Geometric numerical integration illustrated by the Störmer/Verlet method*. Acta Numerica 12, 399–450. [作者页面](https://www.unige.ch/~hairer/preprints/gniverlet.html)
4. Rein, H. & Spiegel, D. S. (2015). *IAS15: A fast, adaptive, high-order integrator for gravitational dynamics*. MNRAS 446, 1424–1437. [原文](https://arxiv.org/abs/1409.4779)
5. SciPy `solve_ivp` [官方文档](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html)：本次独立参考实际使用 DOP853，配置、版本与数据校验值保存在 `data/threebody_reference/metadata.json`。
